In [1]:
!pip install transformers
!pip install accelerate

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 53.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.3/190.3 KB 12.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 66.5 MB/s eta 0:00:00
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.7/199.7 KB 11.5 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM 
import re
from tqdm import tqdm
import os

# Device 할당
# GPU USE
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# TOKENIZER LOAD
TOKENIZER = AutoTokenizer.from_pretrained(
  'kakaobrain/kogpt', revision='KoGPT6B-ryan1.5b-float16',
  bos_token='[BOS]', eos_token='[EOS]', unk_token='[UNK]', pad_token='[PAD]', mask_token='[MASK]')

# MODEL LOAD float 16 Version KOGPT3
model = AutoModelForCausalLM.from_pretrained(
  'kakaobrain/kogpt', revision='KoGPT6B-ryan1.5b-float16',  
  pad_token_id=TOKENIZER.pad_token_id,
  torch_dtype=torch.float16, low_cpu_mem_usage=True
).to(device='cuda', non_blocking=True)

# Evaluate Mode Setting
model.eval()
print('Inference is Ready')

In [ ]:
# 예제 Text
sample_prompt ='''
'제9호 태풍 ‘마이삭’이 당초 예상대로 경남을 관통하면서 곳곳에 생채기를 냈다.  다만 인명피해 등 큰 피해는 발생하지 않았다.   
3일 경남도 등에 따르면 마이삭은 2003년 9월 경남에서만 65명이 숨지거나 실종되는 등 
최악의 인명과 재산 피해를 냈던 태풍 매미와 이동 경로가 비슷해 창원·거제·통영 등 해안가 도시를 중심으로 비상이 걸렸다.     
2003년 당시에는 창원시 마산합포구의 경우 매미가 몰고 온 강풍과 해일 등이 경남대학교 앞 중심 상가(댓거리)를 덮치면서 18명이 목숨을 잃었다.  
이곳을 비롯해 해운동과 월영동은 태풍 뿐 아니라 집중호우 때면 도로가 바닷물에 잠기는 일이 적지 않았다.  하지만 이번에는 달랐다.  
창원시는 하루 전날 해안가 저지대 주민에게 대피 권고를 내리고 지하상가 등에는 영업 중단을 요청하는 등 태풍에 대비했다.    
또 올해 완공된 서항지구배수펌프장이 가동을 시작하면서 월영동과 해운동 등의 배수 능력이 커졌다.  
이 배수펌프장은 1분당 2174t의 빗물을 퍼 올려 배수로를 통해 마산만으로 내보낼 수 있다.  
실제 마이삭 북상과 만조 시간에 맞춰 2일 오후 7시부터 3일 오전 2시까지 서항지구배수펌프장을 가동했다.  
이 시간 마산만 수위가 평소 만조 때보다 20㎝ 높은 최대 258㎝까지 올라갔으나 침수 피해는 발생하지 않았다.  
창원시 관계자는 “258㎝까지 수위가 상승하면 경남대 앞 도로까지 침수되는데 이번에는 배수펌프장 가동으로 침수피해가 발생하지 않았다”고 말했다.  
또 다른 상습 침수 지역인 진해구 용원지역은 일부 침수 피해가 발생했으나 시가지가 물에 잠기는 일은 없었다.   
인명 피해 등은 없었지만, 곳곳에서 태풍으로 인한 생채기가 났다.  강풍으로 전선 등이 끊어지면서 경남 10개 시·군 2만1912가구에 정전이 발생했다.  
3일 오전 7시까지 절반도 복구가 되지 않았는데 한국전력 측은 이날 중 전력 복구를 마칠 예정이다.   
창원시 진해구 용원동에서는 주택 외벽이 무너져 주차 차량을 덮쳤고 양산시 상동면에서는 주택 지붕이 날아갔다.  
통영시에서는 교회 첨탑이 무너지거나 어선 1척이 침몰했다.  통영시와 양산시, 고성군에서는 가로수가 쓰러졌다.  
양산 에덴벨리 골프장 인근에 세워져 있던 높이 70m 정도의 풍력기도 파손된 것으로 전해졌다.  
농작물 피해는 현재 경남도에서 조사 중이다.  양식장은 창원과 거제에서 12건의 피해 신고가 들어와 자세한 내용을 파악 중이다. '
'''

In [ ]:
# 정규표현식을 활용한 전처리 및 요약문을 위한 Keyword(한줄 요약) 기입
def preprocess(data):
    data = re.sub("[^A-Za-z0-9가-힣]",' ',data) 
    return data+' 한줄 요약 :'

In [ ]:
sample_prompt = preprocess(sample_prompt)

In [ ]:
tokens = TOKENIZER.encode(sample_prompt, return_tensors='pt').to(device='cuda:0', non_blocking=True)

In [ ]:
# Greedy Search
gen_tokens = model.generate(tokens,do_sample=False,max_length=len(tokens[0])+1005)
generated = TOKENIZER.batch_decode(gen_tokens)[0]
decoded = generated.replace(sample_prompt,'')
results = re.sub("[^A-Za-z0-9가-힣]",' ',decoded) 
print(results)

태풍 마이삭이 경남을 관통하면서 곳곳에 생채기를 냈다 인명피해는 없었지만 해안가 저지대 주민에게 대피 권고를 내리고 지하상가 등에는 영업 중단을 요청하는 등 태풍에 대비했다 또 올해 완공된 서항지구배수펌프장이 가동을 시작하면서 월영동과 해운동 등의 배수 능력이 커졌다 EOS 


In [ ]:
# Baem Search Wihtout Penalty
gen_tokens = model.generate(tokens, do_sample=False,num_beams=5,max_length=len(tokens[0])+100)
generated = TOKENIZER.batch_decode(gen_tokens)[0]
decoded = generated.replace(sample_prompt,'')
results = re.sub("[^A-Za-z0-9가-힣]",' ',decoded) 
print(results)


태풍 마이삭이 당초 예상대로 경남을 관통하면서 곳곳에 생채기를 냈다 다만 인명피해 등 큰 피해는 발생하지 않았다 EOS 


In [ ]:
# Beam Search With Penalty
gen_tokens = model.generate(tokens, do_sample=False,num_beams=3,max_length=len(tokens[0])+100,no_repeat_ngram_size=3)
generated = TOKENIZER.batch_decode(gen_tokens)[0]
decoded = generated.replace(sample_prompt,'')
results = re.sub("[^A-Za-z0-9가-힣]",' ',decoded) 
print(results)

태풍 마이삭이 경남에 상륙하면서 창원 거제 등 해안가 저지대에 사는 주민들은 대피를 하는 등 대비를 했고 다행히 인명피해는 없었으나 곳곳에서 피해가 났다  EOS 


In [ ]:
# Top K Search Without Temperature
gen_tokens = model.generate(tokens, do_sample=True,top_k=50,max_length=len(tokens[0])+100)
generated = TOKENIZER.batch_decode(gen_tokens)[0]
decoded = generated.replace(sample_prompt,'')
results = re.sub("[^A-Za-z0-9가-힣]",' ',decoded) 
print(results)

  마이삭 부산 경남 관통  인명피해 는 없어 EOS 


In [ ]:
# Top K Search With Temperature
gen_tokens = model.generate(tokens, do_sample=True,top_k=50, temperature=1.05, max_length=len(tokens[0])+100)
generated = TOKENIZER.batch_decode(gen_tokens)[0]
decoded = generated.replace(sample_prompt,'')
results = re.sub("[^A-Za-z0-9가-힣]",' ',decoded) 
results

'  2003년 태풍 매미  이번 태풍은 이동하는 동선이 비슷해 경남 해안가 저지대 중심으로 집중관리했으나 인명과 재산 피해는 다소 발생했고   일부는 아직 복구되는 중이다   EOS '